In [1]:
!pip install openai scikit-learn numpy pandas

In [19]:
import os
import time
import numpy as np
import pandas as pd
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

# Iske theek niche apni API key aise likhein:
client = OpenAI(api_key="sk-your-openai-api-key-here")

In [11]:
corpus = [
    # Topic 1: Sports
    "The striker scored a spectacular volley in the final minutes of the match.",
    "A forward netted an astonishing goal right before the referee blew full-time.",
    "The basketball team executed an aggressive full-court press defense.",
    "Athletes must maintain rigorous cardiovascular conditioning for endurance.",
    "The championship tournament concluded with an intense overtime shootout.",

    # Topic 2: Technology
    "Engineers deployed a scalable distributed database across cloud servers.",
    "Developers launched an elastic clustered storage system on remote cloud nodes.",
    "Modern neural networks require massive GPU clusters for efficient training.",
    "Open-source software libraries drive rapid iteration in artificial intelligence.",
    "Microservice architectures decouple monolithic backends into modular APIs.",

    # Topic 3: Cooking
    "Whisk the eggs gently with heavy cream before pouring into a warm pan.",
    "Beat the yolks softly with whole milk prior to cooking on low heat.",
    "Simmer the diced tomatoes and minced garlic over low flame for rich flavor.",
    "Baking artisan sourdough bread demands precise hydration and natural yeast fermentation.",
    "Season the seared steak generously with coarse sea salt and cracked black pepper.",

    # Topic 4: Travel
    "Backpackers navigated the winding trails of the alpine mountain range.",
    "Hikers explored the zigzagging paths through steep snowy peaks.",
    "Exploring historic cobblestone alleys reveals centuries of architectural heritage.",
    "Reserving budget accommodations in advance prevents unexpected lodging expenses.",
    "Coastal road trips offer breathtaking panoramic views of the turquoise ocean."
]

true_labels = ["Sports"] * 5 + ["Technology"] * 5 + ["Cooking"] * 5 + ["Travel"] * 5
print(f"Dataset ready! Total sentences: {len(corpus)}")

Dataset ready! Total sentences: 20


In [12]:
EMBEDDING_MODEL = "text-embedding-3-small"
COST_PER_1M_TOKENS = 0.02

# API call ka time measure karna
start_time = time.time()
response = client.embeddings.create(
    input=corpus,
    model=EMBEDDING_MODEL
)
total_time = time.time() - start_time

# Vectors array me convert karein
embedding_vectors = np.array([item.embedding for item in response.data])

# Token aur cost calculation
total_tokens = response.usage.total_tokens
estimated_cost = (total_tokens / 1_000_000) * COST_PER_1M_TOKENS

print("=== Batch Metrics ===")
print(f"Total Sentences: {len(corpus)}")
print(f"Time Taken: {total_time:.3f} seconds")
print(f"Total Tokens: {total_tokens}")
print(f"Estimated Cost: ${estimated_cost:.8f}")
print(f"Embedding Shape: {embedding_vectors.shape}")

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [13]:
!pip install sentence-transformers

In [14]:
import time
from sentence_transformers import SentenceTransformer

# Free and open-source model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

start_time = time.time()
embedding_vectors = embed_model.encode(corpus)
total_time = time.time() - start_time

# Task metrics (OpenAI comparison calculation)
total_tokens = sum(len(s.split()) * 1.3 for s in corpus)
simulated_openai_cost = (total_tokens / 1_000_000) * 0.02

print("=== Batch Metrics ===")
print(f"Total Sentences: {len(corpus)}")
print(f"Time Taken: {total_time:.3f} seconds")
print(f"Total Tokens: ~{int(total_tokens)}")
print(f"Estimated OpenAI Cost: ${simulated_openai_cost:.8f}")
print(f"Embedding Shape: {embedding_vectors.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

=== Batch Metrics ===
Total Sentences: 20
Time Taken: 0.410 seconds
Total Tokens: ~271
Estimated OpenAI Cost: $0.00000543
Embedding Shape: (20, 384)


In [15]:
# TF-IDF Cosine Similarity calculate karna
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)
tfidf_sim_matrix = cosine_similarity(tfidf_matrix)

# Dense Embeddings Cosine Similarity
dense_sim_matrix = cosine_similarity(embedding_vectors)

# 5 Paraphrase Pairs (Words alag, meaning same)
paraphrase_pairs = [
    (0, 1, "Sports: Striker volley vs Forward goal"),
    (5, 6, "Tech: Cloud DB vs Clustered storage"),
    (10, 11, "Cooking: Whisk eggs/cream vs Beat yolks/milk"),
    (15, 16, "Travel: Backpackers alpine vs Hikers snowy peaks"),
    (3, 13, "Conceptual: Athlete conditioning vs Sourdough discipline")
]

print("=== TF-IDF vs Dense Embedding Similarity Comparison ===\n")
for idx1, idx2, label in paraphrase_pairs:
    t_score = tfidf_sim_matrix[idx1, idx2]
    d_score = dense_sim_matrix[idx1, idx2]
    print(f"[{label}]")
    print(f"  Sentence 1: \"{corpus[idx1]}\"")
    print(f"  Sentence 2: \"{corpus[idx2]}\"")
    print(f"  TF-IDF Similarity:       {t_score:.4f}")
    print(f"  Embedding Similarity:   {d_score:.4f}\n")

=== TF-IDF vs Dense Embedding Similarity Comparison ===

[Sports: Striker volley vs Forward goal]
  Sentence 1: "The striker scored a spectacular volley in the final minutes of the match."
  Sentence 2: "A forward netted an astonishing goal right before the referee blew full-time."
  TF-IDF Similarity:       0.0637
  Embedding Similarity:   0.4983

[Tech: Cloud DB vs Clustered storage]
  Sentence 1: "Engineers deployed a scalable distributed database across cloud servers."
  Sentence 2: "Developers launched an elastic clustered storage system on remote cloud nodes."
  TF-IDF Similarity:       0.0873
  Embedding Similarity:   0.5901

[Cooking: Whisk eggs/cream vs Beat yolks/milk]
  Sentence 1: "Whisk the eggs gently with heavy cream before pouring into a warm pan."
  Sentence 2: "Beat the yolks softly with whole milk prior to cooking on low heat."
  TF-IDF Similarity:       0.0691
  Embedding Similarity:   0.6277

[Travel: Backpackers alpine vs Hikers snowy peaks]
  Sentence 1: "Backpac

In [16]:
def embed_and_recommend(query_sentence, corpus_sentences, corpus_embeddings, top_k=3):
    # Query ka embedding nikalna
    query_vec = embed_model.encode([query_sentence])

    # Cosine similarity nikalna
    sims = cosine_similarity(query_vec, corpus_embeddings)[0]

    # Top K similar sentences ke indices
    top_indices = np.argsort(sims)[::-1][:top_k]

    print(f"=== Recommendations for: \"{query_sentence}\" ===")
    for rank, idx in enumerate(top_indices, 1):
        print(f"Rank {rank} (Score: {sims[idx]:.4f}): {corpus_sentences[idx]}")

# Test query run karein
test_query = "Preparing homemade pasta requires fresh dough and simmering sauce."
embed_and_recommend(test_query, corpus, embedding_vectors, top_k=3)

=== Recommendations for: "Preparing homemade pasta requires fresh dough and simmering sauce." ===
Rank 1 (Score: 0.4481): Simmer the diced tomatoes and minced garlic over low flame for rich flavor.
Rank 2 (Score: 0.2854): Beat the yolks softly with whole milk prior to cooking on low heat.
Rank 3 (Score: 0.2481): Whisk the eggs gently with heavy cream before pouring into a warm pan.


In [17]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embedding_vectors)

cluster_df = pd.DataFrame({
    "Topic": true_labels,
    "Cluster": clusters,
    "Sentence": corpus
})

print("=== K-Means Cluster Alignment Matrix ===")
print(pd.crosstab(cluster_df["Topic"], cluster_df["Cluster"]))

=== K-Means Cluster Alignment Matrix ===
Cluster     0  1  2  3
Topic                 
Cooking     0  0  5  0
Sports      5  0  0  0
Technology  0  0  0  5
Travel      0  5  0  0


## Theoretical Explanation: Sparse TF-IDF vs Dense Embeddings

### 1. Representation & Dimensionality
- **TF-IDF (Sparse Vectors):** Dimensionality vocabulary size (|V|) ke barabar hoti hai (thousands of dimensions), jisme mostly values 0 hoti hain. Yeh exact word presence aur statistical rarity par depend karta hai.
- **Dense Embeddings:** Low-dimensional, compact continuous coordinate space (e.g., 384 dimensions) hota hai jisme har position par floating-point values hoti hain.

### 2. Generalization Across Vocabulary
- **Lexical vs Semantic:** TF-IDF tab fail hota hai jab sentences ka meaning same ho par vocabulary different ho (jaise "striker volley" vs "forward goal"), kyunki word overlap na hone se dot product 0 ho jata hai.
- **Contextual Topology:** Dense embeddings transformer networks se context aur distributional semantics capture karte hain. Words bina overlap ke bhi high-dimensional space me geometric proximity share karte hain, isliye paraphrases high cosine similarity score generate karte hain.